# 69장 · FastAPI StreamingResponse와 줄 단위 JSON

답변 문장이 모두 보인 순간에도 done 한 줄이 아직 도착하지 않았을 수 있습니다.

고정 답변을 NDJSON 이벤트로 전송합니다. 같은 스트림을 중간에 끊은 경우와 저장 뒤 완료 이벤트까지 수신한 경우의 화면 상태를 비교합니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 fastapi 0.135.0입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## StreamingResponse의 이벤트 직렬화

NDJSON에서는 JSON 객체 하나 뒤에 줄바꿈을 둡니다. 실제 네트워크 조각의 끝이 줄의 끝과 일치하지 않을 수 있으므로, 클라이언트는 완성된 줄부터 읽습니다. 아래는 /conversations/{cid}/stream 경로의 줄 단위 전송과 소유자 검사를 축약한 로컬 앱입니다.

### 스트림의 시작과 마지막 줄 구별하기

**스트림을 열었습니다.**　시작 이벤트에는 답변이 끝났다는 정보가 없습니다.

```
받은 이벤트: start
```

**이 예에서는 답 한 문장이 한 번에 옵니다.**　화면에 문장이 다 보여도 done을 받기 전에는 수신 측에서 저장 여부와 완료 여부를 확인할 수 없습니다.

```
받은 이벤트: start, delta
```

**완료 이벤트를 받았습니다.**　노트북이 정상 응답을 마지막 줄까지 읽었을 때 확인한 순서입니다. 서버의 대화 기록에 저장된 항목 수도 셀에서 따로 확인합니다.

```
받은 이벤트: start, delta, done
```


### 셀 01 · StreamingResponse 경로와 소유자 검사

start에는 요청 ID가 들어 있습니다. delta는 미리 정한 답변 한 문장 전체를 한 번에 보냅니다.


In [1]:
import json
from fastapi import FastAPI, Header, HTTPException
from fastapi.responses import StreamingResponse
from fastapi.testclient import TestClient

app = FastAPI()
owners = {"chat-a": "course-alice"}
history = {"chat-a": []}
def line(kind, **fields):
    return json.dumps({"type": kind, **fields}, ensure_ascii=False) + "\n"

@app.post("/conversations/{cid}/stream")
def stream(cid: str, body: dict, x_course_token: str = Header()):
    if owners.get(cid) != x_course_token:
        raise HTTPException(404)
    def events():
        yield line("start", request_id="demo-1", llm_token_stream=False)
        answer = "과제함에 제출하세요."
        yield line("delta", text=answer)
        history[cid].append((body["question"], answer))
        yield line("done", answer=answer)
    return StreamingResponse(events(), media_type="application/x-ndjson")

client = TestClient(app)

여기서는 공개 수업용 토큰을 소유자 구분에만 씁니다. 실제 로그인, 외부 모델 토큰 스트리밍, 네트워크 재접속은 구현하지 않았습니다.


### 첫 이벤트보다 앞선 소유자 확인

<p>stream 함수는 먼저 owners.get(cid)의 결과와 요청 헤더를 비교합니다. 여기서 요청이 거절되면 이벤트 생성기에 들어가지 않아 답변 조각이나 대화 이력이 생기지 않습니다.</p>


### delta 이후 이력을 저장하고 done 전송하기

클라이언트가 본 답변 조각, 서버에 생긴 대화 한 건, 완료 신호의 순서를 확인하세요.

1. 소유자 확인 · X-Course-Token = course-alice
2. start 전송 · request_id = demo-1
3. delta 전송 · text = 과제함에 제출하세요.
4. history 추가 · len(history["chat-a"]) = 1
5. done 전송 · 수신한 이벤트 = start · delta · done


### 셀 02 · 줄 단위 JSON과 완료 이벤트

질문 한 건을 보내고 이벤트 종류를 받은 순서대로 확인하세요.


In [2]:
with client.stream("POST", "/conversations/chat-a/stream",
                   headers={"X-Course-Token": "course-alice"},
                   json={"question": "과제는 어디에 내나요?"}) as response:
    events = [json.loads(record) for record in response.iter_lines()]
print([event["type"] for event in events])
print(len(history["chat-a"]))

['start', 'delta', 'done']
1


끝까지 수신한 첫 요청에서 history에 한 건이 생깁니다. 같은 셀을 다시 실행하면 이전 기록에 새 턴이 추가됩니다. 반면 다음 과제의 partial은 이미 받은 목록을 잘라낸 값이라서 서버 이력은 그대로입니다.


#### 답변 일부가 도착한 시점

<mark>delta는 답변의 일부이고 done을 읽어야 마지막 이벤트까지 받은 것입니다.</mark> 연결이 중간에 끊겨도 화면에는 글자가 남을 수 있습니다. 받은 줄의 종류를 순서대로 확인하면 보이는 문장과 완료 여부를 따로 표시할 수 있습니다.

선수 개념 · 68장 FastAPI 요청 모델과 POST 엔드포인트

#### delta 이후 이력을 저장하고 done 전송하기

```mermaid
flowchart TD
nowner["소유자 확인"]
  nstart["start 전송"]
  ndelta["delta 전송"]
  nsave["history 추가"]
  ndone["done 전송"]
  nowner --> nstart
  nstart --> ndelta
  ndelta --> nsave
  nsave --> ndone
```

클라이언트가 본 답변 조각, 서버에 생긴 대화 한 건, 완료 신호의 순서를 확인하세요.

### NDJSON 한 줄과 이벤트 하나

<p>line은 JSON 객체 하나에 줄바꿈을 붙입니다. 응답을 받는 쪽은 iter_lines로 한 줄씩 읽고 json.loads로 복원합니다. 이번 delta는 이미 정해진 답변 전체 한 조각이며 모델 토큰이 아닙니다.</p>


### 저장한 뒤 보내는 done

<p>서버는 delta를 보낸 다음 history에 질문과 답을 추가하고 done을 보냅니다. 전체 스트림을 끝까지 읽은 셀에서 이력이 한 건인 이유입니다.</p>


## 수신한 본문과 완료 이벤트 구분

저장 뒤 발송한 완료 신호를 클라이언트가 받지 못할 수도 있습니다. 문장이 마침표로 끝났다는 이유만으로 저장 여부를 확정하면 안 됩니다.


### 과제 03 · 부분 스트림의 화면 상태

수신 기록의 마지막 done이 빠졌습니다. 화면이 완료를 확인할 수 있는지 검사하세요.

먼저 끝까지 받은 events에서 마지막을 잘라 수신자가 본 앞부분만 재현합니다. 여기서 완료 신호를 못 봤다고 해서 서버 저장도 실패했다고 결론 내릴 수는 없습니다. 이 노트북에서는 history의 저장 기록을 따로 확인하세요.


In [3]:
partial = events[:2]
# 이벤트 목록에 done이 있는지 검사하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>partial = events[:2]
print(any(item[&#x27;type&#x27;] == &#x27;done&#x27; for item in partial))</code></pre>
<p>실행 결과</p><pre><code>False</code></pre>
<p>False입니다. 이는 클라이언트가 완료 신호를 보지 못했다는 뜻입니다. 서버 이력의 저장 여부는 대화를 다시 조회해야 확인할 수 있습니다.</p>
</details>


### 부분 수신과 저장 여부

<p>partial = events[:2]는 완료한 응답의 앞 두 이벤트만 따로 보는 관찰입니다. 이 목록에 done이 없으므로 화면은 완료를 확인할 수 없습니다. 이 슬라이스가 이미 저장된 history를 취소하지는 않습니다.</p>


### 과제 04 · 다른 소유자의 조회 제한

같은 대화에 다른 소유자의 토큰을 보내세요. 응답 코드와 요청 전후 대화 이력의 길이를 함께 확인하면 거절된 요청이 저장까지 진행했는지 알 수 있습니다.


In [4]:
before = len(history['chat-a'])
# course-bob으로 같은 경로에 질문을 보내고 상태 코드와 이력 변화를 출력하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>before = len(history[&quot;chat-a&quot;])
denied = client.post(&quot;/conversations/chat-a/stream&quot;,
                     headers={&quot;X-Course-Token&quot;: &quot;course-bob&quot;},
                     json={&quot;question&quot;: &quot;제출 위치&quot;})
print(denied.status_code, len(history[&quot;chat-a&quot;]) - before)</code></pre>
<p>실행 결과</p><pre><code>404 0</code></pre>
<p>404는 요청을 거절한 결과이고 0은 새 대화 턴의 수입니다. 상태 코드만 검사했다면 거절된 질문이 이력에 기록되는 오류를 놓쳤을 겁니다.</p>
</details>


### 대화 조회에 쓰는 수업용 헤더

<p>소유 표에서 chat-a의 값은 course-alice입니다. 경로에 같은 chat-a가 들어 있어도 헤더의 값이 다르면 이 예제는 404를 반환합니다. 이 고정 문자열은 수업용 식별 값입니다.</p>


## 잘려 도착한 NDJSON 응답에서 답변 복원하기

서버가 start, delta, done을 차례로 보냈더라도 수신할 때 바이트가 잘리는 위치는 다를 수 있습니다. 이 실습은 httpx의 줄 단위 읽기를 통해 화면에 보여 줄 문자열과 저장할 시점을 정합니다.

### 셀 08 · 한 글자의 UTF-8 바이트 중간에서 자르기

먼저 셀 01을 실행해 줄바꿈을 붙이는 line 함수를 준비하세요. 서버 응답을 파일처럼 미리 준비합니다. 과의 첫 바이트 뒤에서 잘라 두 조각으로 보내고 iter_lines가 내주는 첫 delta를 읽습니다.

In [5]:
import httpx
import json

wire = (line("start") + line("delta", text="과제") +
        line("delta", text="함") + line("done", answer="과제함")).encode("utf-8")
cut = wire.index("과".encode("utf-8")) + 1

def training_stream():
    return httpx.Response(200, headers={"Content-Type": "application/x-ndjson; charset=utf-8"},
                          content=iter([wire[:cut], wire[cut:]]))

chunks = [json.loads(row) for row in training_stream().iter_lines()]
print(chunks[1]["type"], chunks[1]["text"])

delta 과제


delta 과제가 나옵니다. httpx가 두 조각의 바이트를 디코딩하고 줄바꿈까지 모은 뒤 한 레코드를 건넸습니다. 평범한 바이트 조각마다 JSON 파서를 호출하면 첫 조각에서 문자열 중간이 끊깁니다.

### 과제 09 · 완료 직전 끊긴 스트림에서 임시 답 보존하기

delta의 text를 누적하고 done이 실제로 도착한 경우에만 완료 값을 True로 돌려주는 함수를 작성하세요. 마지막 레코드 직전에 중단한 경우와 끝까지 읽은 경우를 각각 검사하세요.

In [6]:
# render_events(records)는 (화면 문자열, 완료 여부)를 반환합니다.
# chunks[:-1]과 chunks를 한 번씩 넣어 같은 문자열의 완료 상태를 비교하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>def render_events(records):
    text = &quot;&quot;
    finished = False
    for event in records:
        if event[&quot;type&quot;] == &quot;delta&quot;:
            text += event[&quot;text&quot;]
        elif event[&quot;type&quot;] == &quot;done&quot;:
            finished = True
    return text, finished

print(render_events(chunks[:-1]))
print(render_events(chunks))</code></pre>
<p>실행 결과</p><pre><code>(&#x27;과제함&#x27;, False)
(&#x27;과제함&#x27;, True)</code></pre>
<p>두 줄에 모두 과제함이 보이지만 첫 줄은 False입니다. 화면에 단어가 완성돼 보여도 서버의 완료 이벤트를 받았는지는 따로 기록해야 합니다. 이 입력은 네트워크 지연을 측정하는 용도가 아닙니다.</p>
</details>

<mark>답변 글자의 개수로 스트림의 종료를 결정할 수 없습니다.</mark> 이 예제에서는 마지막 델타와 함께 모든 글자가 이미 도착합니다. 그다음 done을 놓치면 읽을 수 있는 문자열이 남아도 완료 표시는 보류합니다.

기존 TestClient 경로의 iter_lines 결과도 같은 함수에 넘겨, 고정된 바이트 입력과 실제 앱 경로의 처리를 이어 보세요.

## 돌아보기

두 사람이 같은 답변 문장을 보았더라도 한쪽만 done까지 수신할 수 있습니다. 화면의 완료 표시에는 수신 이벤트를, 실제 저장 여부에는 서버의 이력을 근거로 삼습니다.
